# Multimodal Deep Learning — Text + Image Classification
Run all cells top-to-bottom. Update the **Paths** cell before training.

## 0. Configuration & Paths

In [7]:
import os

# ── Edit these paths to match your environment ──────────────────────────
DATASET_JSON_PATH = "/content/multimodal_dataset.json"
IMAGES_DIR = "/content/assets/images"
SAVED_MODELS_DIR = "/content/saved_models"
EVAL_OUTPUT_PATH = "/content/evaluation_results.json"
# ────────────────────────────────────────────────────────────────────────

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
print('Saved models directory:', os.path.abspath(SAVED_MODELS_DIR))

Saved models directory: /content/saved_models


In [8]:
import os

print(os.getcwd())

/content


In [9]:
print(os.listdir())

['.config', 'multimodal_dataset.json', 'assets.zip', 'saved.zip', 'saved_models', 'sample_data']


In [10]:
!unzip -q /content/assets.zip -d /content/

In [11]:
import os
print(os.listdir("/content"))

['.config', 'multimodal_dataset.json', 'assets.zip', 'assets', 'saved.zip', 'saved_models', 'sample_data']


## 1. Dataset Preparation

In [12]:
import json
import torch
from torch.utils.data import Dataset
from PIL import Image
from sklearn.preprocessing import LabelEncoder
import torchvision.transforms as transforms


class MultimodalDataset(Dataset):
    def __init__(self, json_file_path, images_dir=IMAGES_DIR, transform=None):
        """
        Multimodal Dataset for text and image classification.

        Args:
            json_file_path (str): Path to the JSON file containing the dataset.
            images_dir (str): Root directory where images are stored.
            transform (callable, optional): Optional transform to be applied on the image.
        """
        with open(json_file_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)

        self.images_dir = images_dir

        for item in self.data:
            item['text'] = item.get('title', '') + ' ' + item.get('description', '')

        labels = [item['label'] for item in self.data]
        self.label_encoder = LabelEncoder()
        encoded_labels = self.label_encoder.fit_transform(labels)
        for i, item in enumerate(self.data):
            item['encoded_label'] = encoded_labels[i]

        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
            ])
        else:
            self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']

        image_path = os.path.join(self.images_dir, item['image_name'])
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f'Error loading image from {image_path}: {e}')
            image = Image.new('RGB', (224, 224), color=(128, 128, 128))

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(item['encoded_label'], dtype=torch.long)

        return {'text': text, 'image': image, 'label': label}


dataset = MultimodalDataset(DATASET_JSON_PATH)
print(f'Dataset size: {len(dataset)}')
sample = dataset[0]
print(f'Sample text : {sample["text"][:100]}...')
print(f'Image shape : {sample["image"].shape}')
print(f'Label       : {sample["label"]}')
print(f'Classes     : {list(dataset.label_encoder.classes_)}')

Dataset size: 4724
Sample text :  ...
Image shape : torch.Size([3, 224, 224])
Label       : 4
Classes     : [np.str_('accessories'), np.str_('bottoms'), np.str_('footwear'), np.str_('other'), np.str_('tops')]


In [13]:
import torch
print(torch.cuda.is_available())

True


In [14]:
dataset = MultimodalDataset(DATASET_JSON_PATH)
print(len(dataset))

4724


## 2. Text Processing

In [15]:
class TextProcessor:
    def __init__(self, max_len=100):
        """
        Simple Text Processor for GRU input preparation.

        Args:
            max_len (int): Maximum length for padding sequences.
        """
        self.max_len = max_len
        self.vocab = None
        self.word_to_idx = {}
        self.idx_to_word = {}

    def fit(self, texts):
        all_words = set()
        for text in texts:
            words = self._tokenize(text)
            all_words.update(words)

        self.word_to_idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx_to_word = {0: '<PAD>', 1: '<UNK>'}

        for word in sorted(all_words):
            idx = len(self.word_to_idx)
            self.word_to_idx[word] = idx
            self.idx_to_word[idx] = word

        self.vocab = self.word_to_idx

    def _tokenize(self, text):
        return text.lower().split()

    def transform(self, text):
        tokens = self._tokenize(text)
        indices = [self.word_to_idx.get(token, self.word_to_idx['<UNK>']) for token in tokens]

        if len(indices) < self.max_len:
            indices += [self.word_to_idx['<PAD>']] * (self.max_len - len(indices))
        else:
            indices = indices[:self.max_len]

        return torch.tensor(indices, dtype=torch.long)

    def encode(self, text):
        return self.transform(text)


texts = [item['text'] for item in dataset.data]
text_processor = TextProcessor(max_len=100)
text_processor.fit(texts)
print(f'Vocabulary size: {len(text_processor.vocab)}')
sample_encoded = text_processor.transform(texts[0])
print(f'Encoded shape  : {sample_encoded.shape}')

Vocabulary size: 2
Encoded shape  : torch.Size([100])


## 3. Text Encoder

In [16]:
import torch.nn as nn


class TextGRUEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, output_dim=256, num_layers=1, dropout=0.0):
        """
        PyTorch model for encoding text using Embedding + GRU + FC.

        Args:
            vocab_size (int): Size of the vocabulary.
            embed_dim (int): Dimension of embedding layer.
            hidden_dim (int): Hidden dimension of GRU.
            output_dim (int): Output dimension (fixed-size vector).
            num_layers (int): Number of GRU layers.
            dropout (float): Dropout rate.
        """
        super(TextGRUEncoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        hidden = hidden[-1]
        output = self.fc(hidden)
        return output


vocab_size = len(text_processor.vocab)
TEXT_DIM = 256

text_gru_encoder = TextGRUEncoder(vocab_size=vocab_size, output_dim=TEXT_DIM)
print(f'TextGRUEncoder vocab_size={vocab_size}, output_dim={TEXT_DIM}')

dummy_text = torch.randint(0, vocab_size, (2, 100))
print(f'Test output shape: {text_gru_encoder(dummy_text).shape}')

TextGRUEncoder vocab_size=2, output_dim=256
Test output shape: torch.Size([2, 256])


## 4. Image Encoder

In [17]:
from torchvision import models


class ImageEncoder(nn.Module):
    def __init__(self, output_dim=256):
        """
        Image Encoder using pretrained ResNet18.

        Args:
            output_dim (int): Output feature dimension.
        """
        super(ImageEncoder, self).__init__()
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, output_dim)

    def forward(self, x):
        return self.resnet(x)


IMAGE_DIM = 256

image_encoder = ImageEncoder(output_dim=IMAGE_DIM)
print(f'ImageEncoder output_dim={IMAGE_DIM}')

dummy_img = torch.randn(2, 3, 224, 224)
print(f'Test output shape: {image_encoder(dummy_img).shape}')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 134MB/s]


ImageEncoder output_dim=256
Test output shape: torch.Size([2, 256])


## 5. Fusion Model

In [18]:
class MultimodalFusionModel(nn.Module):
    def __init__(self, text_dim=256, image_dim=256, hidden_dim=512, num_classes=5, dropout=0.5):
        """
        Multimodal Fusion Model for text + image classification.

        Args:
            text_dim (int): Text feature dimension.
            image_dim (int): Image feature dimension.
            hidden_dim (int): Hidden layer dimension.
            num_classes (int): Number of output classes.
            dropout (float): Dropout rate.
        """
        super(MultimodalFusionModel, self).__init__()

        input_dim = text_dim + image_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.dropout1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.dropout2 = nn.Dropout(dropout)

        self.fc3 = nn.Linear(hidden_dim // 2, num_classes)

        self.relu = nn.ReLU()

    def forward(self, text_features, image_features):
        x = torch.cat((text_features, image_features), dim=1)
        x = self.relu(self.fc1(x))
        x = self.dropout1(x)
        x = self.relu(self.fc2(x))
        x = self.dropout2(x)
        return self.fc3(x)


num_classes = len(dataset.label_encoder.classes_)

fusion_model = MultimodalFusionModel(
    text_dim=TEXT_DIM,
    image_dim=IMAGE_DIM,
    num_classes=num_classes
)
print(f'MultimodalFusionModel: text_dim={TEXT_DIM}, image_dim={IMAGE_DIM}, num_classes={num_classes}')

t_feat = torch.randn(4, TEXT_DIM)
i_feat = torch.randn(4, IMAGE_DIM)
print(f'Test output shape: {fusion_model(t_feat, i_feat).shape}')

MultimodalFusionModel: text_dim=256, image_dim=256, num_classes=5
Test output shape: torch.Size([4, 5])


## 6. Training

In [19]:
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
import pickle


def train_model(
    json_file_path,
    num_epochs=10,
    batch_size=32,
    learning_rate=0.001,
    device='cpu'
):
    device = torch.device(device)

    dataset = MultimodalDataset(json_file_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    all_texts = [item['text'] for item in dataset.data]
    text_encoder = TextProcessor(max_len=100)
    text_encoder.fit(all_texts)
    vocab_size = len(text_encoder.vocab)

    text_model = TextGRUEncoder(vocab_size=vocab_size, output_dim=TEXT_DIM).to(device)
    image_model = ImageEncoder(output_dim=IMAGE_DIM).to(device)
    n_classes = len(dataset.label_encoder.classes_)
    fusion = MultimodalFusionModel(
        text_dim=TEXT_DIM, image_dim=IMAGE_DIM, num_classes=n_classes
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        list(text_model.parameters()) +
        list(image_model.parameters()) +
        list(fusion.parameters()),
        lr=learning_rate
    )

    for epoch in range(num_epochs):
        text_model.train()
        image_model.train()
        fusion.train()

        epoch_loss = 0
        all_preds, all_labels = [], []

        for batch in dataloader:
            batch_texts  = batch['text']
            images  = batch['image'].to(device)
            labels  = batch['label'].to(device)

            text_inputs = torch.stack(
                [text_encoder.transform(t) for t in batch_texts]
            ).to(device)

            text_features  = text_model(text_inputs)
            image_features = image_model(images)
            logits = fusion(text_features, image_features)

            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        print(f'Epoch {epoch+1}/{num_epochs} | '
              f'Loss: {epoch_loss/len(dataloader):.4f} | '
              f'Accuracy: {acc:.4f}')

    print('\nTraining completed!')

    torch.save(text_model.state_dict(),
               os.path.join(SAVED_MODELS_DIR, 'text_gru_encoder.pth'))
    torch.save(image_model.state_dict(),
               os.path.join(SAVED_MODELS_DIR, 'image_encoder.pth'))
    torch.save(fusion.state_dict(),
               os.path.join(SAVED_MODELS_DIR, 'fusion_model.pth'))

    with open(os.path.join(SAVED_MODELS_DIR, 'label_encoder.pkl'), 'wb') as f:
        pickle.dump(dataset.label_encoder, f)
    with open(os.path.join(SAVED_MODELS_DIR, 'text_encoder.pkl'), 'wb') as f:
        pickle.dump(text_encoder, f)

    print('Models and encoders saved to:', SAVED_MODELS_DIR)
    return text_model, image_model, fusion, text_encoder, dataset.label_encoder


text_model, image_model, fusion_model, text_encoder, label_encoder = train_model(
    json_file_path=DATASET_JSON_PATH,
    num_epochs=10,
    batch_size=16,
    learning_rate=0.001,
    device='cuda'
)

Epoch 1/10 | Loss: 0.4600 | Accuracy: 0.8184
Epoch 2/10 | Loss: 0.3403 | Accuracy: 0.8567
Epoch 3/10 | Loss: 0.3954 | Accuracy: 0.8484
Epoch 4/10 | Loss: 0.3656 | Accuracy: 0.8529
Epoch 5/10 | Loss: 0.2102 | Accuracy: 0.8857
Epoch 6/10 | Loss: 0.2641 | Accuracy: 0.8715
Epoch 7/10 | Loss: 0.2236 | Accuracy: 0.8802
Epoch 8/10 | Loss: 0.1745 | Accuracy: 0.8988
Epoch 9/10 | Loss: 0.2009 | Accuracy: 0.9253
Epoch 10/10 | Loss: 0.2029 | Accuracy: 0.9229

Training completed!
Models and encoders saved to: /content/saved_models


## 7. Evaluation

In [20]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)


def load_components(device):
    """Load saved models and encoders from SAVED_MODELS_DIR."""
    with open(os.path.join(SAVED_MODELS_DIR, 'text_encoder.pkl'), 'rb') as f:
        text_enc = pickle.load(f)
    with open(os.path.join(SAVED_MODELS_DIR, 'label_encoder.pkl'), 'rb') as f:
        label_enc = pickle.load(f)

    vocab_size  = len(text_enc.vocab)
    n_classes   = len(label_enc.classes_)

    t_model = TextGRUEncoder(vocab_size=vocab_size, output_dim=256).to(device)
    i_model = ImageEncoder(output_dim=256).to(device)
    f_model = MultimodalFusionModel(
        text_dim=256, image_dim=256, num_classes=n_classes
    ).to(device)

    t_model.load_state_dict(torch.load(
        os.path.join(SAVED_MODELS_DIR, 'text_gru_encoder.pth'), map_location=device))
    i_model.load_state_dict(torch.load(
        os.path.join(SAVED_MODELS_DIR, 'image_encoder.pth'), map_location=device))
    f_model.load_state_dict(torch.load(
        os.path.join(SAVED_MODELS_DIR, 'fusion_model.pth'), map_location=device))

    t_model.eval()
    i_model.eval()
    f_model.eval()

    return text_enc, label_enc, t_model, i_model, f_model


def evaluate_model(json_file_path, batch_size=4, device='cpu'):
    device = torch.device(device)
    dataset    = MultimodalDataset(json_file_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    text_enc, label_enc, t_model, i_model, f_model = load_components(device)

    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            batch_texts = batch['text']
            images  = batch['image'].to(device)
            labels  = batch['label'].to(device)

            text_inputs = torch.stack(
                [text_enc.transform(t) for t in batch_texts]
            ).to(device)

            text_features  = t_model(text_inputs)
            image_features = i_model(images)
            logits = f_model(text_features, image_features)

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    accuracy  = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1        = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    cm        = confusion_matrix(all_labels, all_preds)
    report    = classification_report(
        all_labels, all_preds,
        target_names=label_enc.classes_,
        output_dict=True
    )

    results = {
        'accuracy':                float(accuracy),
        'precision':               float(precision),
        'recall':                  float(recall),
        'f1_score':                float(f1),
        'confusion_matrix':        cm.tolist(),
        'class_names':             list(label_enc.classes_),
        'classification_report':   report,
        'total_samples':           len(all_labels),
        'correct_predictions':     int(sum(p == l for p, l in zip(all_preds, all_labels)))
    }
    return results


results = evaluate_model(DATASET_JSON_PATH, batch_size=4, device='cpu')

print('=' * 50)
print('EVALUATION RESULTS')
print('=' * 50)
print(f'Accuracy  : {results["accuracy"]:.4f}')
print(f'Precision : {results["precision"]:.4f}')
print(f'Recall    : {results["recall"]:.4f}')
print(f'F1 Score  : {results["f1_score"]:.4f}')
print(f'Total     : {results["total_samples"]}')
print(f'Correct   : {results["correct_predictions"]}')

with open(EVAL_OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved to: {EVAL_OUTPUT_PATH}')

EVALUATION RESULTS
Accuracy  : 0.9505
Precision : 0.9641
Recall    : 0.9505
F1 Score  : 0.9480
Total     : 4724
Correct   : 4490

Results saved to: /content/evaluation_results.json


## 8. Inference

In [21]:
def run_inference(text_input, image_path, device='cpu'):
    """
    Run inference on a single text + image pair.

    Args:
        text_input (str): Raw text input.
        image_path (str): Path to the image file.
        device (str): 'cpu' or 'cuda'.

    Returns:
        str: Predicted class label.
    """
    device = torch.device(device)

    text_enc, label_enc, t_model, i_model, f_model = load_components(device)

    text_tensor = text_enc.transform(text_input).unsqueeze(0).to(device)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    try:
        image = Image.open(image_path).convert('RGB')
        image_tensor = transform(image).unsqueeze(0).to(device)
    except Exception as e:
        print(f'[WARN] Image load failed: {e}. Using blank image.')
        image_tensor = torch.zeros((1, 3, 224, 224)).to(device)

    with torch.no_grad():
        text_features  = t_model(text_tensor)
        image_features = i_model(image_tensor)
        logits = f_model(text_features, image_features)
        predicted_idx = torch.argmax(logits, dim=1).item()
        predicted_class = label_enc.classes_[predicted_idx]

    return predicted_class


# ── Edit these to test your own sample ──────────────────────────────────
TEXT_SAMPLE  = 'Nike Swim Hydroguard Men\'s Short-Sleeve Top. Nike.com'
IMAGE_SAMPLE = os.path.join(IMAGES_DIR, 'tshirts_tops/27c2f4ca3f06753511820a9cca353ec2.jpg')
# ────────────────────────────────────────────────────────────────────────

prediction = run_inference(TEXT_SAMPLE, IMAGE_SAMPLE)

print('=' * 50)
print('INFERENCE RESULT')
print('=' * 50)
print(f'Text       : {TEXT_SAMPLE}')
print(f'Image      : {IMAGE_SAMPLE}')
print(f'Prediction : {prediction}')

INFERENCE RESULT
Text       : Nike Swim Hydroguard Men's Short-Sleeve Top. Nike.com
Image      : /content/assets/images/tshirts_tops/27c2f4ca3f06753511820a9cca353ec2.jpg
Prediction : tops


# additions w kda

In [22]:
import os
print(os.listdir("/content/saved_models"))

['text_gru_encoder.pth', 'fusion_model.pth', 'text_encoder.pkl', 'image_encoder.pth', 'label_encoder.pkl']


In [23]:
!zip -r saved_models.zip /content/saved_models

  adding: content/saved_models/ (stored 0%)
  adding: content/saved_models/text_gru_encoder.pth (deflated 8%)
  adding: content/saved_models/fusion_model.pth (deflated 7%)
  adding: content/saved_models/text_encoder.pkl (deflated 11%)
  adding: content/saved_models/image_encoder.pth (deflated 7%)
  adding: content/saved_models/label_encoder.pkl (deflated 43%)


In [24]:
from google.colab import files
files.download('/content/saved_models.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>